# ULMS infercnvpy for the revision of the paper
- Uses adata.raw (log normalized data) that is not batch corrected, with all immune cells as a reference

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import infercnvpy as cnv
from pathlib import Path
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpacnv

/labs/delitto/james/.envs/jpa_infercnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas: 2.2.3
numpy: 2.2.4
scanpy: 1.11.1
infercnv: 0.6.0


In [2]:
# version control
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scanpy:", sc.__version__)
print("seaborn:", sns.__version__)
print("infercnv:", cnv.__version__)

numpy: 2.2.4
pandas: 2.2.3
scanpy: 1.11.1
seaborn: 0.13.2
infercnv: 0.6.0


In [3]:
# Set up input and output directories
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent
print(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / 'objects'
print(DATA_DIR)

output_dir = jpacnv.create_output_dir(PROJECT_DIR, 'cnv', change_dir=True)

/oak/stanford/groups/longaker/ULMS/revision/scRNAseq
/oak/stanford/groups/longaker/ULMS/revision/scRNAseq/objects
Created output directory /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/cnv
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/cnv


In [4]:
N_JOBS = 16
WINDOW_SIZE = 500

# Running the infercnvpy pipeline

In [5]:
# careful with the input directory
adata = sc.read_h5ad(DATA_DIR / 'annotated.h5ad')
adata

AnnData object with n_obs × n_vars = 223655 × 2000
    obs: 'batch', 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'doublet', 'doublet_score', '_scvi_batch', '_scvi_labels', 'leiden1_1', '_scvi_raw_norm_scaling', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'leiden1_6', 'leiden1_7', 'leiden1_8', 'leiden1_9', 'leiden2_0', 'celltype'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'N_scVI', 'X_pca', '_scvi_manager_uuid', '_scvi_uuid', 'batch_colors', 'celltype_colors', 'dendrogram_leiden1_1', 'dendrogram_leiden1_2', 'dendrogram_leiden1_3', 'dendrogram_leiden1_4', 'dendrogram_leiden1_5

In [6]:
# this is key - need to use all the genes, not just the HVG, for cnv analysis
# adata.raw contains the lognormalized counts prior to HVG selection.
rawdata = adata.raw.to_adata()
ref = PROJECT_DIR / 'ref/GRCh38-2024-A.txt'
sample_adata = sc.read_h5ad(PROJECT_DIR / 'preprocessed/Batch01_pp_singlet_adata.h5ad')
ensg_map = sample_adata.var['gene_ids'].to_dict()
del sample_adata
rawdata = jpacnv.gof_format(rawdata, ref=ref, ensg_map=ensg_map)

                            ensg  chromosome    start      end
index                                                         
DDX11L2          ENSG00000290825        chr1    11869    14409
MIR1302-2HG      ENSG00000243485        chr1    29554    31109
FAM138A          ENSG00000237613        chr1    34554    36081
ENSG00000290826  ENSG00000290826        chr1    57598    64116
OR4F5            ENSG00000186092        chr1    65419    71585
...                          ...         ...      ...      ...
ENSG00000277836  ENSG00000277836  KI270728.1  1270984  1271271
ENSG00000278633  ENSG00000278633  KI270731.1    10598    13001
ENSG00000276017  ENSG00000276017  KI270734.1    72411    74814
ENSG00000278817  ENSG00000278817  KI270734.1   131494   137392
ENSG00000277196  ENSG00000277196  KI270734.1   138082   161852

[37964 rows x 4 columns]


In [7]:
types = rawdata.obs["celltype"].unique()
print(*types)
rawdata.var.loc[:, ["ensg", "chromosome", "start", "end"]].head()

Tumor Endothelial Pericyte Myeloid T_and_NK Fibroblast pDC B Plasma Mast RBC Adipocyte Epithelial


,ensg,chromosome,start,end
index,,,,
DDX11L2,ENSG00000290825,chr1,11869,14409
MIR1302-2HG,ENSG00000243485,chr1,29554,31109
FAM138A,ENSG00000237613,chr1,34554,36081
ENSG00000290826,ENSG00000290826,chr1,57598,64116
OR4F5,ENSG00000186092,chr1,65419,71585


In [ ]:
# this time keep the X chromosome and exclude only the Y chromosome
# Increased the window size to more genes to make the contrast between tumor and immune cells more obvious and less noisy
immune_types = ["B", "Myeloid", "T_and_NK", "Mast", "Plasma", "pDC"]
cnv.tl.infercnv(rawdata, 
                reference_key="celltype", 
                reference_cat=immune_types,
                exclude_chromosomes=('chrY',), 
                window_size=WINDOW_SIZE, 
                n_jobs=N_JOBS
               ) 

In [ ]:
rawdata = jpacnv.cnv_plot(rawdata, resolution=1.7)

In [ ]:
ad_path = Path(output_dir / 'rawdata_with_cnv.h5ad')
rawdata.write_h5ad(ad_path)
del rawdata

# Making a table of median CNV scores per cluster

In [ ]:
adata = sc.read_h5ad(output_dir / 'rawdata_with_cnv.h5ad')
adata

In [ ]:
leiden_key = 'leiden1_7'
median_per_cluster = adata.obs.groupby(leiden_key, observed=True)['cnv_score'].median()
adata.obs['cnv_score_cluster_median'] = adata.obs[leiden_key].map(median_per_cluster)
adata.obs['cnv_score_cluster_median']

In [ ]:
# Create a bar plot for genes by counts
median_per_cluster = median_per_cluster.reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(x=leiden_key, y='cnv_score', data=median_per_cluster)
plt.title('Median CNV score for each cluster')
plt.ylabel('median CNV score per cluster')
plt.tight_layout()  # Adjust layout to prevent clipping
plt.savefig(f'median_cnv_score_{leiden_key}_barplot.png')
plt.show()

# Plotting individual clusters on the CNV UMAP

In [ ]:
adata = sc.read_h5ad(output_dir / 'rawdata_with_cnv.h5ad')
adata

In [ ]:
leiden_key = 'leiden1_7'
cats = adata.obs[leiden_key].cat.categories.tolist()
print(*cats)

In [ ]:
for cat in cats:
    target_cluster = cat
    adata.obs['is_target_cluster'] = (adata.obs[leiden_key] == target_cluster)
    cnv.pl.umap(adata, 
                color='is_target_cluster', 
                groups=[False, True], 
                palette=['lightgray', 'red'], 
                size=10, 
                title=f'Highlight Leiden cluster {target_cluster}', 
                save=f'cnv_umap_{target_cluster}.png'
               )

In [ ]:
celltypes = adata.obs["celltype"].cat.categories.tolist()
print(*celltypes)

In [ ]:
for celltype in celltypes:
    target_annotation = celltype
    adata.obs['is_target_annotation'] = (adata.obs['celltype'] == target_annotation)
    cnv.pl.umap(adata, 
                color='is_target_annotation', 
                groups=[False, True], 
                palette=['lightgray', 'red'], 
                size=10, 
                title=f'Highlight Leiden cluster {target_annotation}',
                save=f'cnv_umap_{target_annotation}.png'
               )

In [ ]:
sc.pl.umap(adata, color=['leiden1_7', 'cnv_score'])

# Plotting cnv score on tumor subset umap

In [ ]:
# # careful with the input directory
# path = Path('/oak/stanford/groups/longaker/ULMS/analysis_v3/objects/tumor_subset_raw.h5ad')
# adata = sc.read_h5ad(path)
# adata

In [ ]:
# # careful with the input directory
# path = Path(output_dir / 'rawdata_with_cnv.h5ad')
# cnv = sc.read_h5ad(path)
# cnv

In [ ]:
# adata.obs['cnv_score'] = cnv.obs.loc[adata.obs.index, 'cnv_score']
# adata.obs

In [ ]:
# sc.pl.umap(adata, color='cnv_score', save='cnv_score_tumor_subset.png')